In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_daily_inventory
# PURPOSE   : silver.pos_transactions + silver.warehouse_inventory
#             + silver.product_master + silver.store_inventory
#             → gold.fact_daily_inventory
# LAYER     : Gold (daily inventory fact with store shelf perspective)
# FREQUENCY : Daily (after all silver tables complete)
# WRITE     : Dynamic partition overwrite by transaction_date
# GRAIN     : store_id + product_id + transaction_date
#
# LOGIC:
#   daily_sales  → POS aggregated to store+product+date grain
#   wh_agg       → warehouse stock aggregated to product grain
#   prod_slim    → product master current version (is_current=True)
#   store_inv    → store shelf stock (current_quantity, expiry_date)
#   join all four → compute stock KPIs + health status
#
# KEY DIFFERENCES FROM fact_inventory_full:
#   fact_inventory_full  → warehouse perspective (stock across all warehouses)
#   fact_daily_inventory → store shelf perspective (stock on actual shelf)
#   days_on_hand here uses current_quantity (shelf) not available_stock (warehouse)
#   is_stockout / is_low_stock based on shelf quantity not warehouse quantity
#
# NOTE:
#   inventory_date = transaction_date cast to TimestampType (matches PBI schema)
#   reorder_level / max_stock are DoubleType — avg() across warehouses returns Double
#   stock_value_at_cost  = current_quantity * cost_price (shelf value)
#   stock_value          = available_stock * cost_price (warehouse value)
#   reorder_needed_qty   = max(0, reorder_level - current_quantity) (shelf-based)
#   reorder_quantity     = max(0, reorder_level - available_stock) (warehouse-based)
# =============================================================================

# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json

_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"

sys.path.insert(0, PROJECT_ROOT)

from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import GOLD_FACT_DAILY_INVENTORY_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when,
    sum as spark_sum, count, avg,
    greatest, coalesce, broadcast,
    to_date, to_timestamp
)
from pyspark.sql.types import LongType

with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)

POS_TABLE      = cfg["tables"]["silver_pos_transactions"]
WH_TABLE       = cfg["tables"]["silver_warehouse_inventory"]
PM_TABLE       = cfg["tables"]["silver_product_master"]
SINV_TABLE     = cfg["tables"]["silver_store_inventory"]
TARGET_TABLE   = cfg["tables"]["gold_fact_daily_inventory"]
PIPELINE       = "gold_fact_daily_inventory"

In [0]:
# ── Read + Join + Compute + Write ─────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, POS_TABLE)

try:
    # ── INCREMENTAL READ — POS ────────────────────────────────────────────────
    # Watermark on pos_transactions — process only new transactions
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)

    if last_run_time:
        pos_df = (
            spark.read.table(POS_TABLE)
            .filter(col("processed_at") > lit(last_run_time))
        )
    else:
        pos_df = spark.read.table(POS_TABLE)

    rows_read = pos_df.count()
    print(f"[READ] {rows_read} new POS transaction rows")

    if rows_read == 0:
        print("[SKIP] No new rows to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0)
        raise SystemExit(0)

    # ── DAILY SALES AGGREGATION ───────────────────────────────────────────────
    # Aggregate POS to grain: store_id + product_id + transaction_date
    # transaction_ts is TimestampType in silver — derive date from it
    daily_sales = (
        pos_df
        .withColumn("transaction_date", to_date(col("transaction_ts")))
        .groupBy("store_id", "product_id", "transaction_date")
        .agg(
            spark_sum(col("quantity").cast("long")).alias("units_sold"),
            spark_sum(col("total_amount").cast("double")).alias("revenue"),
            count("*").cast("long").alias("transaction_count"),
            avg(col("unit_price").cast("double")).alias("avg_unit_price")
        )
    )

    # ── WAREHOUSE AGGREGATION ─────────────────────────────────────────────────
    # Aggregate to product grain — across all warehouses
    # current_stock / available_stock → sum (total across warehouses)
    # reorder_level / max_stock → avg (his logic, returns Double, keep as-is)
    wh_agg = (
        spark.read.table(WH_TABLE)
        .groupBy("product_id")
        .agg(
            spark_sum(col("current_stock").cast("long")).alias("current_stock"),
            spark_sum(col("available_stock").cast("long")).alias("available_stock"),
            avg(col("reorder_level").cast("double")).alias("reorder_level"),
            avg(col("max_stock").cast("double")).alias("max_stock")
        )
    )

    # ── PRODUCT MASTER ────────────────────────────────────────────────────────
    # is_current=True — guaranteed one row per product from SCD2 silver
    prod_slim = (
        spark.read.table(PM_TABLE)
        .filter(col("is_current") == True)
        .select(
            "product_id",
            col("cost_price").cast("double"),
            col("selling_price").cast("double"),
            "category",
            "supplier_id"
        )
    )

    # ── STORE INVENTORY SNAPSHOT ──────────────────────────────────────────────
    # Latest shelf snapshot per store+product from silver.store_inventory
    # current_quantity = IntegerType in silver — keep as-is matches his schema
    store_inv = (
        spark.read.table(SINV_TABLE)
        .select(
            "store_id",
            "product_id",
            col("current_quantity"),
            "expiry_date"
        )
    )

    # ── JOIN ──────────────────────────────────────────────────────────────────
    # daily_sales is the driving table — left joins preserve all sales records
    # even if warehouse/product/store data is missing for some products
    fact_raw = (
        daily_sales
        .join(broadcast(wh_agg),   "product_id",              "left")
        .join(broadcast(prod_slim), "product_id",             "left")
        .join(store_inv,           ["store_id", "product_id"], "left")
    )

    # ── DERIVED KPIs ──────────────────────────────────────────────────────────
    df = (
        fact_raw

        # stock_value: warehouse available stock value
        .withColumn("stock_value",
            round(
                coalesce(col("available_stock"), lit(0)).cast("double") *
                coalesce(col("cost_price"), lit(0.0)),
                2
            )
        )

        # days_on_hand: store shelf quantity / daily units sold
        # 999 when no sales (avoids division by zero)
        .withColumn("days_on_hand",
            when(col("units_sold") > 0,
                round(
                    coalesce(col("current_quantity"), lit(0)).cast("double") /
                    col("units_sold").cast("double"),
                    1
                )
            ).otherwise(lit(999.0))
        )

        # reorder_quantity: warehouse-based (how much to order to fill warehouse)
        .withColumn("reorder_quantity",
            greatest(
                lit(0.0),
                coalesce(col("reorder_level"), lit(0.0)) -
                coalesce(col("available_stock"), lit(0)).cast("double")
            )
        )

        # is_stockout: store shelf is empty
        .withColumn("is_stockout",
            coalesce(col("current_quantity"), lit(0)) == 0
        )

        # is_low_stock: store shelf has some stock but <= 20 units
        .withColumn("is_low_stock",
            (coalesce(col("current_quantity"), lit(0)) > 0) &
            (coalesce(col("current_quantity"), lit(0)) <= 20)
        )

        # stock_value_at_cost: store shelf value (shelf qty * cost)
        .withColumn("stock_value_at_cost",
            round(
                coalesce(col("current_quantity"), lit(0)).cast("double") *
                coalesce(col("cost_price"), lit(0.0)),
                2
            )
        )

        # reorder_needed_qty: shelf-based (how much needed on shelf)
        .withColumn("reorder_needed_qty",
            greatest(
                lit(0.0),
                coalesce(col("reorder_level"), lit(0.0)) -
                coalesce(col("current_quantity"), lit(0)).cast("double")
            )
        )

        # inventory_health_status: shelf-based classification
        .withColumn("inventory_health_status",
            when(coalesce(col("current_quantity"), lit(0)) == 0,
                "Stockout")
            .when(coalesce(col("current_quantity"), lit(0)) <= 20,
                "Low Stock")
            .when(
                coalesce(col("current_quantity"), lit(0)).cast("double") >
                coalesce(col("max_stock"), lit(0.0)) * 0.9,
                "Overstock")
            .otherwise("Healthy")
        )

        # inventory_date: transaction_date as timestamp — matches PBI schema
        .withColumn("inventory_date",
            col("transaction_date").cast("timestamp"))

        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))

        .select([f.name for f in GOLD_FACT_DAILY_INVENTORY_SCHEMA.fields])
    )

    rows_written = df.count()

    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("transaction_date")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")

    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)

except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col

spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)

print("Row count:", spark.read.table(TARGET_TABLE).count())

spark.read.table(TARGET_TABLE) \
    .groupBy("inventory_health_status").count().show()

spark.read.table(TARGET_TABLE) \
    .select("store_id", "product_id", "transaction_date",
            "units_sold", "current_quantity", "days_on_hand",
            "inventory_health_status") \
    .limit(5).show(truncate=False)

spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics").show(truncate=False)